# Analisi D FULL v2: PPC D1 originale e D2 con variabilità tra profili

**Correzione della v1:** D1 aveva lo stesso modello, ma un gate PPC diverso dall'originale.
Qui si ripristina il controllo della media per geometria, mantenendo i controlli aggiuntivi
su media/deviazione standard per replica come diagnostiche separate. Non si nascondono i loro fallimenti.
D2 v2 modella invece livelli, intensità e rumore distinti per profilo, con κ condivisi.

**Eseguire tutto in Colab CPU nell'account che contiene i Parquet FULL.** Il notebook monta il
Drive dell'account della sessione, non usa il connettore Drive di Codex. Non servono repository,
Chronos, nuove raccolte o GPU. I risultati empirici appariranno dopo l'esecuzione.

**D1 è conservato:** regressione bayesiana di `log(z_norm)` sugli indicatori delle griglie,
quattro modelli per modalità e confronto LOO. **D2 è una nuova variante esplorativa:**
regressione degli stessi profili su due pettini di depressioni con spaziature libere.
La vecchia regressione di `f1` e `delta_hat` non viene eseguita in questo notebook.

Si leggono gli stessi `02_collapse.parquet` e `02_sites.parquet`. Il secondo è soltanto un
riferimento storico: nessun filtro sui minimi, soglia 0.75/0.90, esclusione dei siti condivisi o
gate di dieci siti entra nel nuovo D2. L'assenza di informazione deve emergere nella posteriore.

Gli output sono in una cartella separata, identificata da hash di input, codice e configurazione.
Una nuova esecuzione riprende soltanto i checkpoint compatibili. Ogni fit viene salvato subito.

## 1. Dipendenze e impostazioni FULL

La cella installa in Colab la linea PyMC 5 / ArviZ 0.22 per mantenere coerenti le API.
Se queste librerie erano già importate con versioni diverse, riavviare il runtime dopo
l'installazione ed eseguire tutto. D1 usa 4 catene, 1.000 warmup e 1.000 draw per catena,
con due catene contemporanee. Sono 12 fit D1, comprese le sinusoidi pure come riferimento.
D2 analizza separatamente KernelSynth e TSMixup.

In [ ]:
from __future__ import annotations
import os,sys,json,hashlib,platform,importlib.util,subprocess,gc,time,inspect
from pathlib import Path
from types import SimpleNamespace
try:
    IS_COLAB=importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IS_COLAB=False
if IS_COLAB:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','pymc>=5.20,<6',
        'arviz==0.22.0','numpy>=1.26,<3','pandas>=2,<4','scipy>=1.11,<2','pyarrow','h5netcdf','h5py'])
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
from IPython.display import display,Markdown
assert pm.__version__.split('.')[0]=='5' and hasattr(az,'InferenceData'), 'Runtime incompatibile: riavviare dopo installazione.'

SOURCE_DATA_DIR=os.environ.get('D_FULL_SOURCE','/content/drive/MyDrive/patchAliasing/full/d3_15model_v1/data')
OUTPUT_ROOT=os.environ.get('D_FULL_OUTPUT','')
D1_REUSE_SOURCE=os.environ.get('D_FULL_D1_REUSE','')
REQUIRE_FULL=True
SEED=42
DRAWS=1000; TUNE=1000; CHAINS=4; CORES=2
D1_MODES=('pure','kernelsynth','tsmixup')
D2_MODES=('kernelsynth','tsmixup')
FS=512.; BAND=(2.,250.); COMB_FLOOR=.01; GRID_TOL_HZ=1.
KAPPA_RANGE=(.25,2.)
GRID_POINTS=801
MAX_GRID_POINTS=1601
MAX_ADAPTIVE_ROUNDS=8
WIDTHS_HZ=(1.,.5,2.)
PRIMARY_WIDTH=1.
D2_PRIOR_SCALE=2.; D2_A0=3.; D2_B0=.5
PREDICTIVE_DRAWS=1000
MODEL_VERSION='D-full-profile-v2-curve-nuisance'
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':.18})
print('PyMC',pm.__version__,'ArviZ',az.__version__,'CPU cores',CORES)

## 2. Parquet, provenienza e copertura

Il preflight FULL richiede 15 geometrie, tre modalità, tre repliche e 265 frequenze per curva.
Verifica chiavi, valori finiti, normalizzazione e hash del manifest. Non normalizza nuovamente
i dati salvati e non sostituisce in silenzio una raccolta smoke. Le repliche rimangono righe
distinte; i profili medi servono solo ai grafici. Il modello mantiene la likelihood condizionalmente
indipendente di D1: la verifica predittiva per geometria/replica non prova indipendenza fra frequenze.

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def atomic_json(path, value):
    path=Path(path);temp=path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(value,indent=2,default=lambda x:x.item() if isinstance(x,np.generic) else str(x)),encoding='utf-8')
    temp.replace(path)

if IS_COLAB and SOURCE_DATA_DIR.startswith('/content/drive/') and not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')
DATA_DIR=Path(SOURCE_DATA_DIR).expanduser().resolve()
INPUTS={name:DATA_DIR/f'02_{name}.parquet' for name in ('collapse','sites')}
for path in INPUTS.values():
    if not path.is_file():raise FileNotFoundError(f'Parquet mancante: {path}. Impostare SOURCE_DATA_DIR.')
HASHES={name:sha256(path) for name,path in INPUTS.items()}
manifest_path=DATA_DIR/'collection_manifest.json'
manifest=json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
verified=True
for name,digest in HASHES.items():
    expected=manifest.get('merged',{}).get(name,{}).get('sha256')
    if expected and expected!=digest:raise ValueError(f'Hash incoerente con il manifest: {name}')
    verified &= expected==digest
collapse=pd.read_parquet(INPUTS['collapse']); sites=pd.read_parquet(INPUTS['sites'])
required={'model','P','S','mode','rep','f','z','z_norm'}
assert required.issubset(collapse.columns), f'Colonne mancanti: {required-set(collapse.columns)}'
assert not collapse.duplicated(['model','mode','rep','f']).any(), 'Frequenze duplicate'
assert np.isfinite(collapse[['P','S','rep','f','z','z_norm']].to_numpy(float)).all()
assert (collapse[['z','z_norm']]>=0).all().all()
assert collapse.f.between(*BAND).all() and (collapse.rep>=0).all()
assert (collapse.P>0).all() and (collapse.S>0).all()
assert collapse.groupby('model')[['P','S']].nunique().eq(1).all().all()
median=collapse.groupby(['model','mode','rep']).z.transform('median').clip(lower=1e-12)
assert np.allclose(collapse.z_norm,collapse.z/median,rtol=1e-5,atol=1e-8), 'z_norm non coerente con la normalizzazione originale'
EXPECTED={(8,8),(16,8),(16,12),(16,16),(24,8),(24,12),(24,16),(24,20),(24,24),
          (32,8),(32,12),(32,16),(32,20),(32,24),(32,32)}
coverage=collapse.groupby(['model','P','S','mode','rep']).agg(points=('f','size')).reset_index()
design_complete=(set(map(tuple,collapse[['P','S']].drop_duplicates().to_numpy()))==EXPECTED
    and set(collapse['mode'])=={'pure','kernelsynth','tsmixup'} and set(collapse.rep)=={0,1,2}
    and len(coverage)==135 and coverage.points.eq(265).all())
full_ok=bool(design_complete and verified and manifest.get('status')=='complete'
    and not manifest.get('design',{}).get('config',{}).get('smoke',False)
    and manifest.get('design',{}).get('reportable') is not False)
if REQUIRE_FULL and not full_ok:raise ValueError('Preflight FULL fallito: controllare copertura e manifest, senza avviare i fit.')
assert set(D1_MODES+D2_MODES).issubset(set(collapse['mode']))
collapse=collapse.sort_values(['mode','model','rep','f']).reset_index(drop=True)
display(coverage.groupby('mode').agg(curves=('points','size'),rows=('points','sum')))
print('FULL verificato:',full_ok,'Parquet sites letto solo come riferimento:',len(sites),'righe')
print('D2: tutte le frequenze, senza rilevatore di minimi.')

## 3. D1 originale

Si conserva la factory originale, inclusi prior, floor e tolleranza di 1 Hz:

\[
\log z_i\sim N(\alpha_g+\theta_S I_S(f_i)+\theta_P I_P(f_i),\sigma^2),\quad
\alpha_g,\theta_S,\theta_P\sim N(0,1),\quad\sigma\sim HalfNormal(1).
\]

Le quattro versioni sono entrambe le griglie, solo stride, solo patch e nessuna griglia.
I punti condivisi attivano entrambi gli indicatori. D1 non richiede minimi rilevati.

In [ ]:
def comb_distance(freqs,spacing,band=BAND):
    freqs=np.asarray(freqs,float)
    grid=np.arange(spacing,band[1]+spacing,spacing)
    grid=grid[grid>=band[0]-spacing]
    return np.min(np.abs(freqs[:,None]-grid[None,:]),axis=1)
pl=SimpleNamespace(FS=FS,comb_distance=comb_distance)
def _codes(series):
    """Integer codes plus the ordered level names, for PyMC `coords`."""
    codes, levels = pd.factorize(series)
    return np.asarray(codes), list(map(str, levels))

def model_D1_sites(df: pd.DataFrame, grid: str) -> pm.Model:
    """Model D1, H3 location.  Deliverable Eq. (12).

        log z_g(f) ~ Normal(alpha_g + theta_S 1[f on stride grid] + theta_P 1[f on patch grid], sigma)

    Each frequency is labelled by which predicted grid it falls on (within GRID_TOL_HZ), and the
    model asks whether the token dispersion is systematically lower there. H3 predicts theta_S < 0.

    `grid` selects the labelling: "both" keeps both indicators (the model H3 states), "stride" only
    c*fs/S, "patch" only k*fs/P, "none" neither. The four are fitted on identical observations and
    compared by LOO,
    which is what identifies WHICH parameter generates the sites, the S<P configurations are the
    ones that separate the two families, since on the P=S diagonal the grids coincide.

    This is deliberately the same shape of model as Eq. (10): a linear predictor on a log scale
    with an indicator variable. It replaced an earlier Gaussian-dip "comb" likelihood with free
    depth and width, which estimated two nuisance quantities nothing downstream used and was far
    harder to justify than the claim it was testing.
    """
    g_c, g_l = _codes(df["model"])
    logz = np.log(np.clip(df["z_norm"].to_numpy(float), COMB_FLOOR, None))

    # membership indicators, built from the same helper that defines the grids everywhere else
    on_stride = np.zeros(len(df))
    on_patch = np.zeros(len(df))
    for gi, _name in enumerate(g_l):
        sel = g_c == gi
        P = int(df.loc[sel, "P"].iloc[0]); S = int(df.loc[sel, "S"].iloc[0])
        f = df.loc[sel, "f"].to_numpy(float)
        on_stride[sel] = (pl.comb_distance(f, pl.FS / S) <= GRID_TOL_HZ).astype(float)
        on_patch[sel] = (pl.comb_distance(f, pl.FS / P) <= GRID_TOL_HZ).astype(float)

    coords = {"geometry": g_l, "obs": np.arange(len(logz))}
    with pm.Model(coords=coords) as m:
        alpha_g = pm.Normal("alpha_g", 0.0, 1.0, dims="geometry")   # off-grid level per geometry
        sigma = pm.HalfNormal("sigma", 1.0)
        mu = alpha_g[g_c]
        if grid in ("stride", "both"):
            theta_S = pm.Normal("theta_S", 0.0, 1.0)
            mu = mu + theta_S * on_stride
        if grid in ("patch", "both"):
            theta_P = pm.Normal("theta_P", 0.0, 1.0)
            mu = mu + theta_P * on_patch
        pm.Normal("logz", mu=mu, sigma=sigma, observed=logz, dims="obs")
    return m

## 4. D2 variante: spaziature libere sul profilo continuo

\[
\Delta_{S,g}=\kappa_S f_s/S_g,\quad\Delta_{P,g}=\kappa_P f_s/P_g,
\qquad C(f;\Delta,w)=\exp[-d(f,\{h\Delta:h\ge1\})^2/(2w^2)].
\]
\[
y_{c,i}=\log z_{c,i}\sim N(\alpha_c+\theta_{S,c} C(f_i;\Delta_{S,g},w)
+\theta_{P,c} C(f_i;\Delta_{P,g},w),\sigma_c^2),\quad c=(g,\mathrm{replica}).
\]

**Prior nuovi, espliciti:** `κ_S` e `κ_P` indipendenti Uniform(0.25, 2.0).
Per ogni profilo `c`, `σ_c² ~ InverseGamma(3, 0.5)` (shape, scale) e
`(α_c, θ_S,c, θ_P,c) | σ_c² ~ Normal(0, 4σ_c² I)`, indipendenti tra profili.
I κ sono condivisi fra geometrie e repliche della stessa modalità. I coefficienti possono essere
positivi o negativi: una depressione non è imposta dal prior. Questo prior coniugato permette
di integrare intercetti, coefficienti e varianza **analiticamente**. La posteriore dei due κ
è approssimata mediante quadratura bidimensionale, non mediante MCMC né ottimizzazione.

`κ=1` non è fissato e non riceve una massa puntuale. L'intervallo 0.25–2.0 è un'assunzione
sostanziale; massa vicino ai bordi blocca l'interpretazione e richiede un intervallo più ampio.
La ROPE è [0.9, 1.1], con massa a priori pari a 0.2/1.75, circa 0.1143 per ramo.
Le depressioni hanno larghezza primaria `w=1 Hz`, con sensibilità obbligatoria a 0.5 e 2 Hz.
Non si media tra queste tre scelte né si sceglie quella che favorisce la teoria.

Il modello descrive una depressione attesa comune alle armoniche di ciascun ramo all'interno
di un profilo. Tra profili intensità, livello e rumore sono liberi: non sono stimati prima del fit,
ma integrati con i loro prior. È una modifica del modello motivata dai PPC v1, non un abbassamento dei gate.
Armoniche poco visibili contribuiscono attraverso i valori continui, ma assenze sistematiche,
forme diverse o variazioni di profondità possono rendere questo modello inadeguato.
Per questo si controlla la ricostruzione dei profili. Una posteriore multimodale va letta
con il grafico congiunto: la sola media può cadere fra soluzioni poco probabili.

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import gammaln, logsumexp, ndtr
from scipy.optimize import brentq


def comb_profile(f, base_spacing, kappas, width):
    delta=np.asarray(base_spacing)[:,None]*np.asarray(kappas)[None,:]
    harmonic=np.maximum(1,np.rint(np.asarray(f)[:,None]/delta))
    return np.exp(-.5*((np.asarray(f)[:,None]-harmonic*delta)/width)**2)


def profile_arrays(frame, floor=.01):
    codes,levels=pd.factorize(frame.model,sort=True)
    return dict(y=np.log(np.clip(frame.z_norm.to_numpy(float),floor,None)),
        f=frame.f.to_numpy(float),s=512./frame.S.to_numpy(float),p=512./frame.P.to_numpy(float),
        codes=codes,levels=list(levels))


def quadrature_widths(k):
    k=np.asarray(k,float)
    assert len(k)>=3 and np.all(np.diff(k)>0)
    return np.r_[(k[1]-k[0])/2,(k[2:]-k[:-2])/2,(k[-1]-k[-2])/2]


def profile_surface(frame, kappas, width=1., prior_scale=2., a0=3., b0=.5, kappas_p=None):
    """Independent conjugate nuisance regressions per geometry/replicate, shared kappas.

    For each curve c: beta_c | sigma_c^2 ~ N(0,tau^2 sigma_c^2 I_3),
    sigma_c^2 ~ InvGamma(a0,b0). No frequency/site selection and no plug-in variance.
    Integrate every beta_c and sigma_c exactly, then integrate kappas by quadrature.
    """
    ks=np.asarray(kappas,float);kp=ks if kappas_p is None else np.asarray(kappas_p,float)
    total=np.zeros((len(ks),len(kp)))
    lam=1/prior_scale**2
    for _,curve in frame.groupby(['model','rep'],sort=True):
        a=profile_arrays(curve);y=a['y'];n=len(y)
        cs=comb_profile(a['f'],a['s'],ks,width)
        cp=comb_profile(a['f'],a['p'],kp,width)
        gs=cs.sum(axis=0);gp=cp.sum(axis=0);ys=y.sum();ig=1/(n+lam)
        ss=lam+(cs*cs).sum(axis=0)-gs*gs*ig
        pp=lam+(cp*cp).sum(axis=0)-gp*gp*ig
        sp=cs.T@cp-ig*gs[:,None]*gp[None,:]
        sy=cs.T@y-gs*ig*ys;py=cp.T@y-gp*ig*ys
        det=ss[:,None]*pp[None,:]-sp*sp
        if np.any(det<=0):raise FloatingPointError('Nonpositive precision determinant')
        explained=(pp[None,:]*sy[:,None]**2+ss[:,None]*py[None,:]**2-2*sp*sy[:,None]*py[None,:])/det
        bn=b0+.5*(y@y-ys*ys*ig-explained);an=a0+n/2
        if np.any(bn<=0):raise FloatingPointError('Nonpositive inverse-gamma scale')
        total+=(-n/2*np.log(2*np.pi)-3*np.log(prior_scale)-.5*(np.log(n+lam)+np.log(det))
                +a0*np.log(b0)-an*np.log(bn)+gammaln(an)-gammaln(a0))
    return dict(ks=ks,kp=kp,logml=total,width=np.asarray(width),prior_scale=np.asarray(prior_scale),
        a0=np.asarray(a0),b0=np.asarray(b0),model_variant=np.asarray('independent-curve-nuisance-v2'))


def surface_weights(surface, axis=None, stride=1, offset=0):
    slices=[slice(None),slice(None)]
    if axis is not None:slices[axis]=slice(offset,None,stride)
    ks=surface['ks'][slices[0]];kp=surface['kp'][slices[1]]
    logml=surface['logml'][tuple(slices)]
    logq=logml+np.log(quadrature_widths(ks)[:,None]*quadrature_widths(kp)[None,:])
    return ks,kp,np.exp(logq-logsumexp(logq)),float(logsumexp(logq))


def summarize_surface(surface):
    ks,kp,weight,log_integral=surface_weights(surface);rows=[]
    for axis,branch,k in [(0,'stride',ks),(1,'patch',kp)]:
        marginal=weight.sum(axis=1-axis);mean=np.dot(k,marginal)
        q=np.interp([.025,.5,.975],np.cumsum(marginal),k)
        rope=marginal[np.abs(k-1)<=.1+1e-10].sum()
        dr=[];dm=[];dz=[]
        for offset in [0,1]:
            cs,cp,wc,zc=surface_weights(surface,axis,2,offset)
            kc=cs if axis==0 else cp;mc=wc.sum(axis=1-axis)
            dr.append(abs(rope-mc[np.abs(kc-1)<=.1+1e-10].sum()))
            dm.append(abs(mean-np.dot(kc,mc)));dz.append(abs(log_integral-zc))
        sd=np.sqrt(np.dot((k-mean)**2,marginal))
        edge=marginal[(k<k[0]+.025)|(k>k[-1]-.025)].sum()
        local_step=np.sqrt(np.dot(quadrature_widths(k)**2,marginal))
        ok=max(dr)<=.02 and max(dm)<=.005 and max(dz)<=.05 and sd>=2*local_step and marginal.max()<.25
        rows.append(dict(branch=branch,width_hz=float(surface['width']),mean=mean,q025=q[0],median=q[1],q975=q[2],
            sd=sd,prob_rope=float(rope),edge_mass=float(edge),grid_delta_rope=max(dr),grid_delta_mean=max(dm),
            grid_delta_log_integral=max(dz),max_cell_mass=float(marginal.max()),grid_ok=bool(ok),range_ok=bool(edge<.01)))
    return pd.DataFrame(rows)


def refined_axes(surface,summary):
    ks,kp,w,_=surface_weights(surface);axes=[]
    for axis,k in enumerate([ks,kp]):
        if summary.iloc[axis].grid_ok:
            axes.append(k);continue
        m=w.sum(axis=1-axis);active=m>1e-8
        active[:-1]|=m[1:]>1e-8;active[1:]|=m[:-1]>1e-8
        intervals=active[:-1]|active[1:]
        axes.append(np.unique(np.r_[k,(k[:-1]+k[1:])[intervals]/2]))
    return axes


def draw_profile_posterior(frame,surface,draws=600,seed=42):
    rng=np.random.default_rng(seed)
    ks,kp,w,_=surface_weights(surface)
    ids=rng.choice(w.size,size=draws,p=w.ravel());si,pi=np.unravel_index(ids,w.shape)
    arr=profile_arrays(frame);n=len(frame)
    means=np.empty((draws,n));sigmas=np.empty((draws,n));betas=[]
    groups=frame.reset_index(drop=True).groupby(['model','rep'],sort=True).indices
    lam=1/float(surface['prior_scale'])**2
    for (model,rep),ix in groups.items():
        y=arr['y'][ix];m=len(ix);sample_beta=np.empty((draws,3))
        cs=comb_profile(arr['f'][ix],arr['s'][ix],ks[si],float(surface['width']))
        cp=comb_profile(arr['f'][ix],arr['p'][ix],kp[pi],float(surface['width']))
        for j in range(draws):
            X=np.column_stack([np.ones(m),cs[:,j],cp[:,j]])
            precision=X.T@X+lam*np.eye(3);chol=np.linalg.cholesky(precision)
            loc=np.linalg.solve(precision,X.T@y)
            bn=float(surface['b0'])+.5*(y@y-loc@(X.T@y));an=float(surface['a0'])+m/2
            sig2=1/rng.gamma(an,1/bn)
            beta=loc+np.sqrt(sig2)*np.linalg.solve(chol.T,rng.normal(size=3))
            means[j,ix]=X@beta;sigmas[j,ix]=np.sqrt(sig2);sample_beta[j]=beta
        betas.append(sample_beta)
    beta=np.stack(betas,axis=1)
    return dict(mu=means,sigma=sigmas,beta=beta,kappa_S=ks[si],kappa_P=kp[pi],
        theta_S_mean=beta[:,:,1].mean(axis=1),theta_P_mean=beta[:,:,2].mean(axis=1),
        noise_rms=np.sqrt(np.mean(sigmas**2,axis=1)))


def geometry_mean_ppc(frame,mu,sigma):
    """Original D1 statistic/strata; analytic Normal-mixture predictive quantiles.

    Integrates observation noise rather than estimating 2.5/97.5 percentiles from 300 replicates.
    sigma is scalar per draw for D1, or a per-observation matrix for D2.
    """
    y=profile_arrays(frame)['y'];rows=[]
    for model,ix in frame.reset_index(drop=True).groupby('model').indices.items():
        locations=mu[:,ix].mean(axis=1)
        scales=sigma/np.sqrt(len(ix)) if sigma.ndim==1 else np.sqrt(np.sum(sigma[:,ix]**2,axis=1))/len(ix)
        low=float(np.min(locations-10*scales));high=float(np.max(locations+10*scales))
        bounds=[brentq(lambda value:float(ndtr((value-locations)/scales).mean())-p,low,high) for p in [.025,.975]]
        observed=float(y[ix].mean())
        rows.append(dict(model=model,statistic='mean',scope='geometry_original',observed=observed,
            lower=bounds[0],upper=bounds[1],inside=bool(bounds[0]<=observed<=bounds[1])))
    return pd.DataFrame(rows)


def predictive_audit(frame,mu,sigma,seed=42):
    rng=np.random.default_rng(seed);y=profile_arrays(frame)['y']
    replicated=mu+rng.normal(size=mu.shape)*(sigma[:,None] if sigma.ndim==1 else sigma)
    rows=[]
    for (model,rep),ix in frame.reset_index(drop=True).groupby(['model','rep']).indices.items():
        for statistic,fun in [('mean',np.mean),('sd',np.std)]:
            observed=fun(y[ix]);low,high=np.quantile(fun(replicated[:,ix],axis=1),[.025,.975])
            rows.append(dict(model=model,rep=int(rep),statistic=statistic,scope='curve_extended',observed=observed,
                lower=low,upper=high,inside=bool(low<=observed<=high)))
    return pd.DataFrame(rows)

## 5. Controlli numerici, checkpoint e costo

Il controllo seguente confronta la formula marginale accelerata con una soluzione matriciale
diretta agli stessi parametri. È una verifica software, non una nuova raccolta o un fit sintetico.
Usa poche righe dei FULL solo per verificare l'algebra. Tutti i fit successivi usano le righe FULL.

La marginalizzazione procede un profilo alla volta. I due assi κ vengono raffinati
separatamente: una posteriore patch larga non costringe a infittire anche l'intero asse stride.
D1 conserva una sola log-likelihood alla volta per LOO, soltanto se manca un fit riutilizzabile.
La precisione numerica di D2 viene verificata anche eliminando alternativamente punti pari/dispari;
se insufficiente, si ripete a 1.601×1.601, poi si raffinano localmente gli intervalli con
massa posteriore apprezzabile, conservando l'intero range. Si verifica anche l'integrale
normalizzante su sottogriglie alternate per ogni asse. Dopo otto raffinamenti un fallimento rimane esplicito.

In [ ]:
def direct_log_marginal(frame,ks,kp,width,scale,a0,b0):
    total=0.
    for _,curve in frame.groupby(['model','rep']):
        a=profile_arrays(curve);y=a['y']
        X=np.column_stack([np.ones(len(y)),comb_profile(a['f'],a['s'],[ks],width)[:,0],
                           comb_profile(a['f'],a['p'],[kp],width)[:,0]])
        A=X.T@X+np.eye(3)/scale**2
        m=np.linalg.solve(A,X.T@y);an=a0+len(y)/2;bn=b0+.5*(y@y-m@(X.T@y))
        total+=(-len(y)/2*np.log(2*np.pi)-3*np.log(scale)-.5*np.linalg.slogdet(A)[1]
            +a0*np.log(b0)-an*np.log(bn)+gammaln(an)-gammaln(a0))
    return total

probe=collapse[collapse['mode'].eq(D2_MODES[0])].groupby('model',group_keys=False).head(5).reset_index(drop=True)
probe_k=np.array([.83,1.,1.17])
probe_surface=profile_surface(probe,probe_k)
for i,ks in enumerate(probe_k):
    for j,kp in enumerate(probe_k):
        np.testing.assert_allclose(probe_surface['logml'][i,j],direct_log_marginal(probe,ks,kp,1.,2.,3.,.5),rtol=1e-9,atol=1e-8)
print('Marginalizzazione verificata; nessun κ fissato a 1.')
CODE_HASH=hashlib.sha256(('37859878c769e862554a93e8c88dd6958a288353cd097d6ba4a734a029beadf9').encode()).hexdigest()
configuration=dict(version=MODEL_VERSION,D1=dict(draws=DRAWS,tune=TUNE,chains=CHAINS,cores=CORES,modes=D1_MODES),
    D2=dict(modes=D2_MODES,kappa_range=KAPPA_RANGE,grid_points=GRID_POINTS,max_grid_points=MAX_GRID_POINTS,max_adaptive_rounds=MAX_ADAPTIVE_ROUNDS,
            widths=WIDTHS_HZ,primary_width=PRIMARY_WIDTH,prior_scale=D2_PRIOR_SCALE,a0=D2_A0,b0=D2_B0),
    floor=COMB_FLOOR,grid_tolerance=GRID_TOL_HZ,seed=SEED,predictive_draws=PREDICTIVE_DRAWS,
    require_full=REQUIRE_FULL,code_hash=CODE_HASH)
assert GRID_POINTS>=5 and GRID_POINTS%2==1 and MAX_GRID_POINTS>=GRID_POINTS
assert KAPPA_RANGE[0]>0 and KAPPA_RANGE[0]<.9<1.1<KAPPA_RANGE[1]
assert PRIMARY_WIDTH in WIDTHS_HZ and D2_PRIOR_SCALE>0 and D2_A0>0 and D2_B0>0
packages={name:__import__(name).__version__ for name in ('numpy','pandas','scipy','pymc','arviz')}
identity=dict(inputs=HASHES,configuration=configuration,packages=packages,manifest_hash=sha256(manifest_path) if manifest else None)
RUN_ID=hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()[:16]
root=Path(OUTPUT_ROOT).expanduser().resolve() if OUTPUT_ROOT else DATA_DIR.parent/'d_full_profiles'
assert root!=DATA_DIR and DATA_DIR not in root.parents, 'Output separati dalla cartella dati'
RUN_DIR=root/RUN_ID;RUN_DIR.mkdir(parents=True,exist_ok=True)
D1_REUSE_DIR=Path(D1_REUSE_SOURCE).expanduser().resolve() if D1_REUSE_SOURCE else DATA_DIR.parent/'d_full_profiles'/'27c183d24ddbadc3'
D1_REUSE_VALID=False
if (D1_REUSE_DIR/'provenance.json').is_file():
    old=json.loads((D1_REUSE_DIR/'provenance.json').read_text())
    old_config=old.get('configuration',{})
    same_d1=json.dumps(old_config.get('D1',{}),sort_keys=True)==json.dumps(configuration['D1'],sort_keys=True)
    D1_REUSE_VALID=bool(old.get('inputs')==HASHES and same_d1
        and old.get('D1_factory_AST')=='f450b67432016b9ca61d75ba385758e0598ef4f73e57f883bcd9166211850741' and old_config.get('floor')==COMB_FLOOR
        and old_config.get('grid_tolerance')==GRID_TOL_HZ and old_config.get('seed')==SEED
        and old.get('full_verified')==full_ok and old.get('status')=='completed'
        and all((D1_REUSE_DIR/f'D1_{mode}_{grid}{ext}').is_file() for mode in D1_MODES
                for grid in ('both','stride','patch','none') for ext in ('.nc','.npz','.json')))
print('Riutilizzo D1 della run precedente:',D1_REUSE_VALID)
if not D1_REUSE_VALID:print('Checkpoint precedenti assenti/incompatibili: i fit mancanti saranno stimati e salvati.')
identity['D1_reused_from']=str(D1_REUSE_DIR) if D1_REUSE_VALID else None
atomic_json(RUN_DIR/'provenance.json',dict(**identity,full_verified=full_ok,source=str(DATA_DIR),status='running'))
print('Cartella risultati e ripresa:',RUN_DIR)
print('D1:',len(D1_MODES)*4,'fit;',DRAWS*CHAINS,'draw ciascuno. D2:',len(D2_MODES)*len(WIDTHS_HZ),'superfici più eventuali raffinamenti.')

In [ ]:
def d1_draws(frame, idata, grid, draws=300):
    post = idata.posterior.stack(sample=('chain','draw'))
    draws = post.sizes['sample'] if draws is None else draws
    index = np.linspace(0,post.sizes['sample']-1, min(draws,post.sizes['sample']),dtype=int)
    post = post.isel(sample=index)
    codes, levels = _codes(frame.model)
    mu = post.alpha_g.transpose('sample','geometry').values[:,codes].copy()
    for branch,column,theta in [('stride','S','theta_S'),('patch','P','theta_P')]:
        if grid not in (branch,'both'):
            continue
        indicator = np.zeros(len(frame))
        for geom in levels:
            mask = frame.model.eq(geom).to_numpy()
            indicator[mask] = pl.comb_distance(frame.loc[mask,'f'],FS/frame.loc[mask,column].iloc[0])<=GRID_TOL_HZ
        mu += post[theta].values[:,None]*indicator[None,:]
    return mu,post.sigma.values


def fit_d1(frame, mode, grid):
    stem=RUN_DIR/f'D1_{mode}_{grid}'
    nc=stem.with_suffix('.nc'); result_file=stem.with_suffix('.json'); loo_file=stem.with_suffix('.npz')
    if D1_REUSE_VALID and not all(p.exists() for p in (nc,result_file,loo_file)):
        import shutil
        for target in (nc,result_file,loo_file):
            shutil.copy2(D1_REUSE_DIR/target.name,target)
        print('D1: posteriori originali riutilizzati da',D1_REUSE_DIR,flush=True)
    if nc.exists() and result_file.exists() and loo_file.exists():
        print('Checkpoint D1:',mode,grid,flush=True)
        return json.loads(result_file.read_text()),dict(np.load(loo_file)),az.from_netcdf(nc)
    model=model_D1_sites(frame,grid)
    with model:
        idata=pm.sample(draws=DRAWS,tune=TUNE,chains=CHAINS,cores=CORES,
                        random_seed=SEED,target_accept=.95,progressbar=True,
                        return_inferencedata=True,idata_kwargs={'log_likelihood':True})
    stats=az.summary(idata,kind='diagnostics')
    diagnostics=dict(rhat_max=float(stats.r_hat.max()),ess_bulk_min=float(stats.ess_bulk.min()),
        ess_tail_min=float(stats.ess_tail.min()),divergences=int(idata.sample_stats.diverging.sum()),
        bfmi_min=float(np.min(az.bfmi(idata))))
    diagnostics['diagnostics_ok']=bool(diagnostics['rhat_max']<=1.01 and diagnostics['ess_bulk_min']>=400
        and diagnostics['ess_tail_min']>=400 and diagnostics['divergences']==0 and diagnostics['bfmi_min']>=.3)
    loo=az.loo(idata,pointwise=True,var_name='logz')
    result=dict(mode=mode,grid=grid,elpd=float(loo.elpd_loo),se=float(loo.se),p_loo=float(loo.p_loo),
        max_pareto_k=float(loo.pareto_k.max()),loo_ok=bool(not loo.warning and (loo.pareto_k<loo.good_k).all()),
        **diagnostics)
    for variable in ('theta_S','theta_P'):
        if variable in idata.posterior:
            values=idata.posterior[variable].values.ravel()
            result[variable+'_prob_negative']=float(np.mean(values<0))
            result[variable+'_q025'],result[variable+'_median'],result[variable+'_q975']=map(float,np.quantile(values,[.025,.5,.975]))
    stats.to_csv(stem.with_name(stem.name+'_diagnostics.csv'))
    arrays=dict(loo_i=loo.loo_i.values,pareto_k=loo.pareto_k.values)
    np.savez_compressed(loo_file,**arrays)
    # LOO is already saved pointwise; retain only small posterior/sampler groups on Drive.
    del idata.log_likelihood
    idata.to_netcdf(nc)
    atomic_json(result_file,result)
    return result,arrays,idata


def plot_profiles(frame, mu, title, path):
    geometries=sorted(frame.model.unique())
    fig,axes=plt.subplots(int(np.ceil(len(geometries)/3)),3,
        figsize=(14,3*int(np.ceil(len(geometries)/3))),squeeze=False,layout='constrained')
    y=profile_arrays(frame)['y']
    for ax,geom in zip(axes.flat,geometries):
        indices=np.flatnonzero(frame.model.eq(geom))
        f=frame.f.to_numpy()[indices]
        freq=np.unique(f)
        observed=np.array([y[indices[f==v]].mean() for v in freq])
        predicted=np.column_stack([mu[:,indices[f==v]].mean(axis=1) for v in freq])
        low,median,high=np.quantile(predicted,[.025,.5,.975],axis=0)
        ax.plot(freq,observed,color='#333333',lw=.8,label='Osservato: media log dispersione')
        ax.plot(freq,median,color='#2171b5',lw=1,label='Media attesa: mediana posteriore')
        ax.fill_between(freq,low,high,color='#2171b5',alpha=.22,label='Intervallo credibile 95% della media')
        ax.set(title=geom,xlabel='Frequenza [Hz]',ylabel='log(z_norm)')
    for ax in list(axes.flat)[len(geometries):]:ax.axis('off')
    axes.flat[0].legend(fontsize=7)
    fig.suptitle(title)
    fig.savefig(path,dpi=130,bbox_inches='tight');plt.show();plt.close(fig)

## 6. Fit D1, confronto LOO e verifica predittiva

Le catene devono avere R-hat ≤1.01, ESS bulk/tail ≥400, nessuna divergenza e BFMI ≥0.3.
Il confronto riporta ELPD e differenza rispetto al migliore con errore standard appaiato.
Non si conclude che i modelli siano diversi dal solo ordinamento dei rank. Pareto-k e
diagnostiche devono essere adeguati. LOO è puntuale sulle frequenze del disegno osservato;
non è validazione su una geometria o replica indipendente.

Per il modello con entrambe le griglie il **PPC originale** controlla la media per geometria,
con copertura minima 90%. Usa tutti i draw disponibili e quantili predittivi calcolati dalla
miscela di Normali, evitando la variabilità aggiuntiva di soli 300 campioni PPC.
I controlli aggiuntivi su media e deviazione standard per geometria/replica sono mostrati
separatamente: i loro fallimenti sono limiti del modello, non un fallimento retroattivo del PPC originale.
Le bande nei grafici rappresentano l'incertezza sulla media attesa; il PPC include anche il rumore.

In [ ]:
D1_RESULTS=[];D1_PPC=[];D1_ORIGINAL_PPC=[];D1_COMPARISONS=[]
for mode in D1_MODES:
    frame=collapse[collapse['mode'].eq(mode)].reset_index(drop=True)
    mode_results=[];mode_loo={}
    for grid in ('both','stride','patch','none'):
        print('D1',mode,grid,flush=True)
        result,loo_arrays,idata=fit_d1(frame,mode,grid)
        if grid=='both':
            all_mu,all_sigma=d1_draws(frame,idata,grid,draws=None)
            original_ppc=geometry_mean_ppc(frame,all_mu,all_sigma);original_ppc.insert(0,'mode',mode)
            D1_ORIGINAL_PPC.append(original_ppc)
            result['ppc_original_ok']=bool(original_ppc.inside.mean()>=.9)
            take=np.linspace(0,len(all_sigma)-1,min(PREDICTIVE_DRAWS,len(all_sigma)),dtype=int)
            mu=all_mu[take];sigma=all_sigma[take];del all_mu,all_sigma
            ppc=predictive_audit(frame,mu,sigma,SEED);ppc.insert(0,'mode',mode)
            D1_PPC.append(ppc)
            result['ppc_extended_ok']=bool(ppc.groupby('statistic').inside.mean().ge(.9).all())
            plot_profiles(frame,mu,f'D1 / {mode} / entrambe le griglie',RUN_DIR/f'D1_{mode}_profiles.png')
            axes=az.plot_trace(idata,var_names=['theta_S','theta_P','sigma'],compact=True)
            plt.gcf().savefig(RUN_DIR/f'D1_{mode}_traces.png',dpi=120,bbox_inches='tight');plt.show();plt.close('all')
            del mu
        mode_results.append(result);mode_loo[grid]=loo_arrays['loo_i']
        D1_RESULTS.append(result)
        display(pd.DataFrame([result]))
        del idata;gc.collect()
    best=max(mode_results,key=lambda r:r['elpd'])['grid']
    comparison=pd.DataFrame(mode_results)
    comparison['elpd_diff_to_best']=[np.sum(mode_loo[g]-mode_loo[best]) for g in comparison.grid]
    comparison['dse']=[np.sqrt(len(frame)*np.var(mode_loo[g]-mode_loo[best],ddof=1)) for g in comparison.grid]
    comparison=comparison.sort_values('elpd',ascending=False)
    D1_COMPARISONS.append(comparison)
    display(comparison[['mode','grid','elpd','elpd_diff_to_best','dse','loo_ok','diagnostics_ok']])
    comparison.to_csv(RUN_DIR/f'D1_{mode}_comparison.csv',index=False)
d1_results=pd.DataFrame(D1_RESULTS);d1_results.to_csv(RUN_DIR/'D1_results.csv',index=False)
d1_ppc=pd.concat(D1_PPC,ignore_index=True);d1_ppc.to_csv(RUN_DIR/'D1_ppc.csv',index=False)
d1_original_ppc=pd.concat(D1_ORIGINAL_PPC,ignore_index=True)
d1_original_ppc.to_csv(RUN_DIR/'D1_ppc_original.csv',index=False)
print('D1: PPC originale, media per geometria')
display(d1_original_ppc.groupby('mode').inside.agg(['sum','count','mean']))
print('D1: diagnostiche aggiuntive per replica, mantenute visibili')
display(d1_ppc.groupby(['mode','statistic']).inside.agg(['sum','count','mean']))

## 7. D2 sui FULL: posteriore congiunta, precisione e larghezza

Si usano tutte le righe dei due generatori, separatamente. Il grafico congiunto serve a vedere
alias e scambi tra le griglie; le marginali mostrano anche il prior uniforme e la ROPE.
`prob_depression` stima la probabilità che la media, a pesi uguali tra profili, dei coefficienti
del ramo sia negativa. Si riportano anche errore Monte Carlo e coefficienti per profilo.
Un κ vicino a 1 con coefficiente non negativo non è evidenza di una griglia di depressioni.
Gli intervalli sono quantili marginali al 95%, non intervalli che isolano un singolo modo.

In [ ]:
def get_surface(frame,mode,width,points,axes=None):
    initial=np.unique(np.r_[np.linspace(*KAPPA_RANGE,points),.9,1.,1.1])
    ks,kp=(initial,initial) if axes is None else axes
    axis_hash=hashlib.sha256(ks.tobytes()+kp.tobytes()).hexdigest()[:10]
    path=RUN_DIR/f'D2v2_{mode}_w{width:g}_{len(ks)}x{len(kp)}_{axis_hash}.npz'
    if path.exists():
        with np.load(path) as data:return {key:data[key] for key in data.files}
    started=time.monotonic()
    result=profile_surface(frame,ks,width,D2_PRIOR_SCALE,D2_A0,D2_B0,kappas_p=kp)
    temp=path.with_name(path.stem+'.tmp.npz');np.savez_compressed(temp,**result);temp.replace(path)
    print('D2 superficie salvata:',mode,'w=',width,'assi=',(len(ks),len(kp)),'secondi=',round(time.monotonic()-started,1),flush=True)
    return result

D2_RESULTS=[];D2_PPC=[];D2_GEOMETRY_PPC=[]
for mode in D2_MODES:
    frame=collapse[collapse['mode'].eq(mode)].reset_index(drop=True)
    for width in WIDTHS_HZ:
        surface=get_surface(frame,mode,width,GRID_POINTS)
        summary=summarize_surface(surface)
        if not summary.grid_ok.all() and MAX_GRID_POINTS>GRID_POINTS:
            print('Raffinamento quadratura:',mode,width,flush=True)
            surface=get_surface(frame,mode,width,MAX_GRID_POINTS)
            summary=summarize_surface(surface)
        for refinement in range(MAX_ADAPTIVE_ROUNDS):
            if summary.grid_ok.all():break
            axes=refined_axes(surface,summary)
            if len(axes[0])*len(axes[1])>12_000_000:
                print('Limite memoria: 12 milioni di nodi congiunti; risultato numerico non validato.');break
            surface=get_surface(frame,mode,width,GRID_POINTS,axes)
            summary=summarize_surface(surface)
        summary.insert(0,'mode',mode)
        summary['grid_points_S']=len(surface['ks']);summary['grid_points_P']=len(surface['kp'])
        posterior=draw_profile_posterior(frame,surface,PREDICTIVE_DRAWS,SEED)
        probabilities=[float(np.mean(posterior[name]<0)) for name in ('theta_S_mean','theta_P_mean')]
        summary['prob_depression']=probabilities
        summary['prob_depression_mcse']=[np.sqrt(p*(1-p)/PREDICTIVE_DRAWS) for p in probabilities]
        geometry_ppc=geometry_mean_ppc(frame,posterior['mu'],posterior['sigma'])
        geometry_ppc.insert(0,'mode',mode);geometry_ppc.insert(1,'width_hz',width);D2_GEOMETRY_PPC.append(geometry_ppc)
        ppc=predictive_audit(frame,posterior['mu'],posterior['sigma'],SEED)
        ppc.insert(0,'mode',mode);ppc.insert(1,'width_hz',width);D2_PPC.append(ppc)
        summary['ppc_ok']=bool(ppc.groupby('statistic').inside.mean().ge(.9).all())
        D2_RESULTS.append(summary)
        summary.to_csv(RUN_DIR/f'D2_{mode}_w{width:g}_summary.csv',index=False)
        pd.DataFrame({key:posterior[key] for key in ('kappa_S','kappa_P','noise_rms','theta_S_mean','theta_P_mean')}).to_csv(
            RUN_DIR/f'D2_{mode}_w{width:g}_draws.csv',index=False)
        nuisance_rows=[]
        for c,((model,rep),ix) in enumerate(frame.groupby(['model','rep'],sort=True).indices.items()):
            row=dict(model=model,rep=int(rep))
            for name,values in [('theta_S',posterior['beta'][:,c,1]),('theta_P',posterior['beta'][:,c,2]),
                                ('sigma',posterior['sigma'][:,ix[0]])]:
                row[name+'_q025'],row[name+'_median'],row[name+'_q975']=np.quantile(values,[.025,.5,.975])
            nuisance_rows.append(row)
        pd.DataFrame(nuisance_rows).to_csv(RUN_DIR/f'D2_{mode}_w{width:g}_profile_parameters.csv',index=False)
        ks,kp,w,_=surface_weights(surface)
        fig,axes=plt.subplots(1,3,figsize=(14,4),layout='constrained')
        density=w/(quadrature_widths(ks)[:,None]*quadrature_widths(kp)[None,:])
        mesh=axes[0].pcolormesh(ks,kp,density.T,shading='auto',cmap='magma')
        fig.colorbar(mesh,ax=axes[0],label='Densità posteriore')
        axes[0].axvline(1,color='cyan',lw=.8);axes[0].axhline(1,color='cyan',lw=.8)
        axes[0].set(xlabel='κ_S',ylabel='κ_P',title='Posteriore congiunta')
        for ax,branch,k,marginal in zip(axes[1:],('stride','patch'),(ks,kp),(w.sum(axis=1),w.sum(axis=0))):
            ax.plot(k,marginal/quadrature_widths(k),color='#2171b5')
            ax.axhline(1/(KAPPA_RANGE[1]-KAPPA_RANGE[0]),color='gray',ls=':',label='Prior')
            ax.axvspan(.9,1.1,color='gray',alpha=.15,label='ROPE ±10%')
            ax.set(xlabel='κ',ylabel='Densità approssimata',title=branch);ax.legend(fontsize=8)
        fig.suptitle(f'D2 / {mode} / larghezza {width:g} Hz')
        fig.savefig(RUN_DIR/f'D2_{mode}_w{width:g}_posterior.png',dpi=140,bbox_inches='tight');plt.show();plt.close(fig)
        display(summary)
        if width==PRIMARY_WIDTH:
            plot_profiles(frame,posterior['mu'],f'D2 / {mode} / larghezza {width:g} Hz',RUN_DIR/f'D2_{mode}_profiles.png')
        del surface,posterior;gc.collect()
d2_results=pd.concat(D2_RESULTS,ignore_index=True);d2_results.to_csv(RUN_DIR/'D2_results.csv',index=False)
d2_ppc=pd.concat(D2_PPC,ignore_index=True);d2_ppc.to_csv(RUN_DIR/'D2_ppc.csv',index=False)
d2_geometry_ppc=pd.concat(D2_GEOMETRY_PPC,ignore_index=True)
d2_geometry_ppc.to_csv(RUN_DIR/'D2_ppc_geometry.csv',index=False)
print('D2: copertura PPC per modalità, larghezza e statistica')
display(d2_ppc.groupby(['mode','width_hz','statistic']).inside.agg(['sum','count','mean']))
print('D2, larghezza primaria: strati che non superano il controllo predittivo')
with pd.option_context('display.max_rows',None):
    display(d2_ppc[np.isclose(d2_ppc.width_hz,PRIMARY_WIDTH) & ~d2_ppc.inside])

## 8. Lettura congiunta e riepilogo

D1 verifica associazione con le griglie fissate; D2 stima le spaziature sotto il nuovo modello.
Non si confronta ELPD di D1 con la likelihood marginale usata internamente da D2.
Una concentrazione nella ROPE è interpretabile solo con effetto medio negativo, risoluzione
numerica adeguata, poca massa ai bordi, PPC adeguato e sensibilità alla larghezza accettabile.
La soglia di probabilità 0.95 è descrittiva per questa variante esplorativa, non una nuova
preregistrazione né una correzione retroattiva delle decisioni originali.

D1 conserva il suo gate storico sulle medie per geometria; i controlli aggiuntivi non vengono
eliminati. D2 v2 mantiene invece il gate più ampio su media e dispersione per replica.
Stimare nuisance per profilo può migliorare questi PPC in-sample: non costituisce una verifica
indipendente delle spaziature e non basta da solo a stabilire H3.

L'intervallo di κ e il prior coniugato delimitano questa analisi. Un risultato favorevole
richiede comunque lettura delle superfici e dei profili; non risolve automaticamente
la dipendenza tra frequenze o la generalizzazione a nuove geometrie.

In [ ]:
lines=['# Analisi D FULL v2: risultati',f'Run: {RUN_ID}',f'FULL verificato: {full_ok}',
       'D1 originale; D2 variante sui profili, senza soglia di rilevazione.',
       '', '## D1']
for comparison in D1_COMPARISONS:
    mode=comparison.iloc[0]['mode'];best=comparison.iloc[0]['grid']
    reliable=bool(comparison.diagnostics_ok.all() and comparison.loo_ok.all())
    primary=comparison[comparison.grid.eq('both')].iloc[0]
    lines.append(f'- {mode}: migliore ELPD {best}; diagnostiche/LOO adeguati: {reliable}; PPC originale (media per geometria): {primary.ppc_original_ok}; diagnostiche aggiuntive per replica: {primary.ppc_extended_ok}.')
original_empirical=d1_original_ppc[d1_original_ppc['mode'].isin(D2_MODES)]
lines.append(f'- PPC originale D1 sui generatori: {int(original_empirical.inside.sum())}/{len(original_empirical)} geometrie; gate >=90%: {bool(original_empirical.inside.mean()>=.9)}.')
lines+=['','## D2']
decision_rows=[]
for (mode,branch),g in d2_results.groupby(['mode','branch']):
    primary=g[np.isclose(g.width_hz,PRIMARY_WIDTH)].iloc[0]
    stable=(g.prob_rope.max()-g.prob_rope.min()<=.1 and g['mean'].max()-g['mean'].min()<=.05)
    numerical=bool(g.grid_ok.all() and g.range_ok.all())
    adequate=bool(g.ppc_ok.all())
    depression=bool(g.prob_depression.ge(.95).all())
    supported=bool(full_ok and stable and numerical and adequate and depression and g.prob_rope.ge(.95).all())
    reasons=[]
    if not numerical:reasons.append('quadratura/range da rivedere')
    if not adequate:reasons.append('PPC non adeguato')
    if not stable:reasons.append('sensibilità alla larghezza')
    if not depression:reasons.append('depressione non sostenuta in tutte le larghezze')
    if not g.prob_rope.ge(.95).all():reasons.append('massa ROPE insufficiente in almeno una larghezza')
    status='SUPPORTO ESPLORATIVO CONDIZIONATO AL MODELLO' if supported else 'SUPPORTO NON STABILITO'
    decision_rows.append(dict(mode=mode,branch=branch,status=status,width_stable=stable,numerical_ok=numerical,
        ppc_ok=adequate,reason='; '.join(reasons)))
    lines.append(f'- {mode}/{branch}: κ mediana {primary["median"]:.4f}, intervallo 95% [{primary.q025:.4f}, {primary.q975:.4f}]; P(ROPE)={primary.prob_rope:.4f}; P(coefficiente<0)={primary.prob_depression:.4f}. {status}. '+ '; '.join(reasons))
decisions=pd.DataFrame(decision_rows);display(decisions)
decisions.to_csv(RUN_DIR/'D2_interpretation.csv',index=False)
for name,path in INPUTS.items():assert sha256(path)==HASHES[name], f'Input cambiato durante la run: {name}'
lines+=['','## Limiti','- D2 è una nuova variante, con prior espliciti e larghezza sottoposta a sensibilità.',
        '- Stessi FULL esplorati in precedenza: nessuna validazione indipendente.',
        '- Le sinusoidi pure in D1 sono un riferimento, non evidenza empirica sui generatori.',
        '- I due Parquet originali sono invariati.']
summary='\n'.join(lines)
(RUN_DIR/'SUMMARY.md').write_text(summary,encoding='utf-8')
atomic_json(RUN_DIR/'provenance.json',dict(**identity,full_verified=full_ok,source=str(DATA_DIR),
    status='completed',original_inputs_unchanged=True,D1_factory_AST='f450b67432016b9ca61d75ba385758e0598ef4f73e57f883bcd9166211850741'))
display(Markdown(summary))
print('Risultati:',RUN_DIR)
print('Restituire questo notebook eseguito; le tabelle principali e i grafici sono inclusi negli output.')

## Riferimenti software

- [PyMC: campionamento e log-likelihood](https://www.pymc.io/projects/docs/en/v5.12.0/api/generated/pymc.sampling.mcmc.sample.html).
- [ArviZ 0.22: LOO e Pareto-k](https://python.arviz.org/en/v0.22.0/api/generated/arviz.loo.html).
- [SciPy: parametrizzazione normal-inverse-gamma](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.normal_inverse_gamma.html).

Il prior coniugato e la forma del pettine di D2 sono scelte della presente variante,
non specifiche imposte da queste librerie. La factory D1 è tratta dagli originali forniti dall'utente.